# Respiratory Sound Classification - Demo Notebook

This notebook demonstrates the complete pipeline for respiratory sound classification using the optimized CNN with Grey Wolf Optimizer.




## 1. Setup and Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import to_categorical
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append('../src')

from preprocessing import AudioPreprocessor, FeatureNormalizer, DataAugmenter
from dataset import RespiratoryDataset
from models import OptimizedCNN, BaselineModels
from grey_wolf_optimizer import GreyWolfOptimizer, get_search_space
from train import ModelTrainer, CrossValidator

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Data Preprocessing Demo

In [ ]:
# Initialize preprocessor
preprocessor = AudioPreprocessor(target_sr=16000, n_mels=40, n_mfcc=13)

# Load a sample audio file
sample_file = '../data/sample_audio.wav'  # Update with actual path

if os.path.exists(sample_file):
    # Preprocess
    features = preprocessor.preprocess(sample_file)
    
    # Visualize
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Waveform
    axes[0, 0].plot(features['audio'])
    axes[0, 0].set_title('Waveform')
    axes[0, 0].set_xlabel('Samples')
    axes[0, 0].set_ylabel('Amplitude')
    
    # ZCR
    axes[0, 1].plot(features['zcr'])
    axes[0, 1].set_title('Zero-Crossing Rate')
    axes[0, 1].set_xlabel('Frames')
    axes[0, 1].set_ylabel('ZCR')
    
    # MFCCs
    im1 = axes[1, 0].imshow(features['mfccs'], aspect='auto', origin='lower', cmap='viridis')
    axes[1, 0].set_title('MFCCs')
    axes[1, 0].set_xlabel('Frames')
    axes[1, 0].set_ylabel('Coefficients')
    plt.colorbar(im1, ax=axes[1, 0])
    
    # Mel-Spectrogram
    im2 = axes[1, 1].imshow(features['mel_spectrogram'], aspect='auto', origin='lower', cmap='viridis')
    axes[1, 1].set_title('Mel-Spectrogram')
    axes[1, 1].set_xlabel('Frames')
    axes[1, 1].set_ylabel('Mel Bins')
    plt.colorbar(im2, ax=axes[1, 1])
    
    plt.tight_layout()
    plt.show()
else:
    print("Sample file not found. Please update the path.")

## 3. Feature Normalization Demo

In [ ]:
# Generate sample data for demonstration
np.random.seed(42)
sample_features = np.random.randn(40, 100) * 50 + 100  # Random features

# Normalize
normalizer = FeatureNormalizer()

# Z-score normalization
z_score_norm, mean, std = normalizer.z_score_normalize(sample_features)

# Min-Max normalization
min_max_norm, min_vals, max_vals = normalizer.min_max_normalize(sample_features)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(sample_features, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original Features')
axes[0].set_xlabel('Time Frames')
axes[0].set_ylabel('Frequency Bins')

axes[1].imshow(z_score_norm, aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('Z-Score Normalized')
axes[1].set_xlabel('Time Frames')
axes[1].set_ylabel('Frequency Bins')

axes[2].imshow(min_max_norm, aspect='auto', origin='lower', cmap='viridis')
axes[2].set_title('Min-Max Normalized')
axes[2].set_xlabel('Time Frames')
axes[2].set_ylabel('Frequency Bins')

plt.tight_layout()
plt.show()

print(f"Original - Mean: {np.mean(sample_features):.2f}, Std: {np.std(sample_features):.2f}")
print(f"Z-Score - Mean: {np.mean(z_score_norm):.2f}, Std: {np.std(z_score_norm):.2f}")
print(f"Min-Max - Min: {np.min(min_max_norm):.2f}, Max: {np.max(min_max_norm):.2f}")

## 4. Grey Wolf Optimizer Demo

In [ ]:
# Get search space
search_space = get_search_space()

print("Hyperparameter Search Space:")
for param, bounds in search_space.items():
    print(f"  {param}: {bounds}")

# Initialize GWO (with small population for demo)
gwo = GreyWolfOptimizer(
    search_space=search_space,
    num_wolves=5,  # Small for demo
    max_iterations=3
)

print("\nGWO initialized successfully!")
print(f"Population size: {gwo.num_wolves}")
print(f"Max iterations: {gwo.max_iterations}")

## 5. Model Architecture Demo

In [ ]:
# Model parameters
input_shape = (40, 40, 1)  # Mel-spectrogram shape
num_classes = 6  # Number of disease categories

# Create Optimized CNN
optimized_cnn = OptimizedCNN(input_shape, num_classes)
model = optimized_cnn.build_model(use_attention=True)

# Display model summary
model.summary()

## 6. Training Demo (Small Scale)

In [ ]:
# Generate dummy data for demonstration
np.random.seed(42)

n_samples = 100
X_dummy = np.random.randn(n_samples, 40, 40, 1)
y_dummy = to_categorical(np.random.randint(0, num_classes, n_samples), num_classes=num_classes)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_dummy, y_dummy, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train model (short training for demo)
trainer = ModelTrainer(
    model_type='optimized_cnn',
    input_shape=input_shape,
    num_classes=num_classes,
    results_dir='../results/demo'
)

history = trainer.train(
    X_train, y_train,
    X_val, y_val,
    epochs=5,  # Short for demo
    batch_size=16,
    use_early_stopping=False
)

print("\nTraining completed!")

In [ ]:
# Evaluate
metrics = trainer.evaluate(X_test, y_test)

print("\nEvaluation Metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Plot training history
trainer.plot_training_history()

## 7. Model Comparison Demo

In [ ]:
# Compare different models
models_to_compare = ['cnn', 'optimized_cnn']
results = {}

for model_type in models_to_compare:
    print(f"\nTraining {model_type.upper()}...")
    
    trainer = ModelTrainer(
        model_type=model_type,
        input_shape=input_shape,
        num_classes=num_classes,
        results_dir=f'../results/demo/{model_type}'
    )
    
    trainer.train(X_train, y_train, X_val, y_val, epochs=5, use_early_stopping=False)
    metrics = trainer.evaluate(X_test, y_test)
    results[model_type] = metrics
    
    print(f"{model_type.upper()} Results:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

In [ ]:
# Visualize comparison
metrics_names = ['accuracy', 'precision', 'recall', 'f1_score']
x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

for i, (model, metrics) in enumerate(results.items()):
    values = [metrics[m] for m in metrics_names]
    ax.bar(x + i*width, values, width, label=model.upper())

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x + width/2)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Conclusion

This notebook demonstrated:
1. Data preprocessing (resampling, ZCR, STFT, MFCCs, Mel-spectrogram)
2. Feature normalization (Z-score, Min-Max)
3. Grey Wolf Optimizer for hyperparameter optimization
4. Optimized CNN architecture with attention mechanism
5. Model training and evaluation
6. Model comparison

For full-scale experiments, use the main script:
```bash
python src/main.py --config config.json
```